<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D1%8B%D0%B9_%D1%82%D0%B5%D0%BB%D0%B5%D0%B1%D0%BE%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Установка необходимых библиотек
!pip install -q fsspec==2025.3.2 transformers datasets peft torch pandas \
    trl bitsandbytes accelerate python-telegram-bot==13.15 nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.4/497.4 kB 26.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.1 MB/s eta 0:00:00

In [2]:
# Импорт необходимых модулей
import os
import torch
import logging
import nest_asyncio
import zipfile
import time
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from telegram import Update, ReplyKeyboardRemove
from telegram.ext import (
    Updater,
    CommandHandler,
    MessageHandler,
    ConversationHandler,
    CallbackContext,
    Filters
)

In [3]:
# Применяем nest_asyncio для избежания ошибки "Cannot close a running event loop"
nest_asyncio.apply()

In [4]:
# Настройка логирования
logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO,
    handlers=[
        logging.FileHandler("bot_log.txt", mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [5]:
def load_model():
    """
    Загружает и распаковывает модель из архива

    Returns:
        tuple: (model, tokenizer) - модель и токенизатор
    """
    model_zip_path = "./qwen2.5-7b-lora-film.zip"
    model_dir = "./qwen2.5-7b-lora-film"

    # Проверяем, есть ли уже распакованная модель
    if not os.path.exists(model_dir):
        # Проверяем, есть ли архив с моделью
        if not os.path.exists(model_zip_path):
            print("Архив с моделью не найден. Пожалуйста, загрузите архив qwen2.5-7b-lora-film.zip")
            uploaded = files.upload()  # Загрузка через диалог

        # Распаковываем архив
        print("Распаковка архива с моделью...")
        with zipfile.ZipFile(model_zip_path, 'r') as zip_ref:
            zip_ref.extractall("./")
        print("Модель распакована успешно!")
    else:
        print("Модель уже распакована.")

    # Загрузка базовой модели и токенизатора
    base_model_id = "Qwen/Qwen2.5-7B-Instruct"
    print("Загрузка токенизатора...")
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print("Загрузка модели...")
    # Квантизация модели для экономии памяти
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    try:
        # Загрузка базовой модели с квантизацией
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        # Загрузка LoRA-адаптера
        print("Загрузка LoRA-адаптера...")
        model = PeftModel.from_pretrained(model, model_dir)

        print("Модель успешно загружена!")
        return model, tokenizer
    except Exception as e:
        logger.error(f"Ошибка при загрузке модели: {e}")
        raise

In [6]:
def create_prompt(name, genre, logline, task_type):
    """
    Создает текстовый промпт для модели

    Args:
        name (str): Название фильма
        genre (str): Жанр фильма
        logline (str): Краткое описание фильма
        task_type (str): Тип задачи ('tagline' или 'annotation')

    Returns:
        str: Промпт для модели
    """
    if task_type == "tagline":
        prompt = f"""Ты — опытный копирайтер, создающий слоганы для фильмов.
Название фильма: {name}
Жанр: {genre}
Краткое описание: {logline}
Сгенерируй яркий и запоминающийся тэглайн, подчеркивающий коммерческий потенциал фильма."""
    elif task_type == "annotation":
        prompt = f"""Ты — профессиональный сценарист. Напиши подробную аннотацию (синопсис) для фильма.
Название фильма: {name}
Жанр: {genre}
Краткое описание: {logline}
Аннотация должна заинтересовать зрителя и подчеркнуть потенциал кассового успеха."""
    else:
        raise ValueError("Неизвестный task_type. Используйте 'tagline' или 'annotation'.")
    return prompt

def generate_text(model, tokenizer, name, genre, logline, task_type):
    """
    Генерирует текст (тэглайн или аннотацию) с помощью модели

    Args:
        model: Модель для генерации текста
        tokenizer: Токенизатор для модели
        name (str): Название фильма
        genre (str): Жанр фильма
        logline (str): Краткое описание фильма
        task_type (str): Тип задачи ('tagline' или 'annotation')

    Returns:
        str: Сгенерированный текст
    """
    try:
        prompt = create_prompt(name, genre, logline, task_type)
        inputs = tokenizer(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=512 if task_type == "annotation" else 32,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(output[0], skip_special_tokens=False)
        response = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
        return response
    except Exception as e:
        logger.error(f"Ошибка при генерации текста: {e}")
        return f"Произошла ошибка при генерации текста: {str(e)}"

In [7]:
# Определяем константы для состояний разговора
NAME, GENRE, LOGLINE, GENERATING = range(4)

# Функция для запуска бота
def start(update: Update, context: CallbackContext) -> int:
    """Начинает разговор и запрашивает название фильма."""
    update.message.reply_text(
        "Привет! Я бот для генерации тэглайнов и аннотаций для фильмов. "
        "Давайте начнем. Пожалуйста, введите название фильма:"
    )
    logger.info(f"Пользователь {update.effective_user.id} начал разговор.")
    return NAME

def name(update: Update, context: CallbackContext) -> int:
    """Сохраняет название фильма и запрашивает жанр."""
    context.user_data["name"] = update.message.text
    logger.info(f"Пользователь {update.effective_user.id} ввел название фильма: {update.message.text}")

    update.message.reply_text(
        "Отлично! Теперь введите жанр фильма:"
    )
    return GENRE

def genre(update: Update, context: CallbackContext) -> int:
    """Сохраняет жанр фильма и запрашивает краткое описание."""
    context.user_data["genre"] = update.message.text
    logger.info(f"Пользователь {update.effective_user.id} ввел жанр фильма: {update.message.text}")

    update.message.reply_text(
        "Хорошо! Теперь введите краткое описание фильма (логлайн):"
    )
    return LOGLINE

def logline(update: Update, context: CallbackContext) -> int:
    """Сохраняет логлайн и начинает генерацию."""
    context.user_data["logline"] = update.message.text
    logger.info(f"Пользователь {update.effective_user.id} ввел логлайн: {update.message.text}")

    update.message.reply_text(
        "Спасибо! Начинаю генерацию тэглайна и аннотации. Это может занять некоторое время..."
    )

    try:
        # Получаем данные из контекста
        name = context.user_data["name"]
        genre = context.user_data["genre"]
        logline = context.user_data["logline"]

        # Генерируем тэглайн
        start_time = time.time()
        tagline = generate_text(model, tokenizer, name, genre, logline, "tagline")
        tagline_time = time.time() - start_time
        context.user_data["tagline"] = tagline
        logger.info(f"Для пользователя {update.effective_user.id} сгенерирован тэглайн за {tagline_time:.2f} сек: {tagline}")

        # Генерируем аннотацию
        start_time = time.time()
        annotation = generate_text(model, tokenizer, name, genre, logline, "annotation")
        annotation_time = time.time() - start_time
        context.user_data["annotation"] = annotation
        logger.info(f"Для пользователя {update.effective_user.id} сгенерирована аннотация за {annotation_time:.2f} сек: {annotation}")

        # Отправляем результаты пользователю
        update.message.reply_text(
            f"Готово! Вот результаты:\n\n"
            f"*Тэглайн:*\n{tagline}\n\n"
            f"*Аннотация:*\n{annotation}\n\n"
            f"Хотите сгенерировать еще один фильм? Отправьте /start",
            parse_mode="Markdown"
        )
    except Exception as e:
        logger.error(f"Ошибка при генерации: {e}")
        update.message.reply_text(
            f"Произошла ошибка при генерации: {str(e)}\n"
            f"Пожалуйста, попробуйте еще раз, отправив /start"
        )

    return ConversationHandler.END

In [8]:
def cancel(update: Update, context: CallbackContext) -> int:
    """Отменяет разговор."""
    logger.info(f"Пользователь {update.effective_user.id} отменил разговор.")
    update.message.reply_text(
        "Генерация отменена. До свидания!",
        reply_markup=ReplyKeyboardRemove()
    )
    return ConversationHandler.END

def get_logs(update: Update, context: CallbackContext) -> None:
    logger.info(f"Пользователь {update.effective_user.id} запросил логи.")
    log_path = "bot_log.txt"
    try:
        if os.path.getsize(log_path) == 0:
            update.message.reply_text("Логи пока пусты.")
            return
        with open(log_path, "rb") as log_file:
            update.message.reply_document(document=log_file)
    except Exception as e:
        logger.error(f"Ошибка при отправке логов: {e}")
        update.message.reply_text(f"Ошибка при отправке логов: {str(e)}")

def help_command(update: Update, context: CallbackContext) -> None:
    """Отправляет сообщение с помощью"""
    update.message.reply_text(
        "Этот бот генерирует тэглайны и аннотации для фильмов.\n\n"
        "Доступные команды:\n"
        "/start - Начать генерацию\n"
        "/cancel - Отменить текущую генерацию\n"
        "/logs - Получить файл логов\n"
        "/help - Показать эту справку"
    )

In [9]:
from getpass import getpass



In [10]:
# Глобальные переменные для модели и токенизатора
model = None
tokenizer = None

# Для запуска бота в Colab
def run_bot():
    """Запускает бота в Colab"""
    global model, tokenizer

    try:
        # Загружаем модель
        model, tokenizer = load_model()

        # Получаем токен бота из переменной окружения или запрашиваем у пользователя
        if "TELEGRAM_BOT_TOKEN" in os.environ:
            token = os.environ["TELEGRAM_BOT_TOKEN"]
        else:
            token = getpass("Введите токен телеграм-бота: ")

        # Создаем обновление бота
        updater = Updater(token)
        dispatcher = updater.dispatcher

        # Создаем обработчик разговора
        conv_handler = ConversationHandler(
            entry_points=[CommandHandler("start", start)],
            states={
                NAME: [MessageHandler(Filters.text & ~Filters.command, name)],
                GENRE: [MessageHandler(Filters.text & ~Filters.command, genre)],
                LOGLINE: [MessageHandler(Filters.text & ~Filters.command, logline)],
            },
            fallbacks=[CommandHandler("cancel", cancel)],
        )

        # Добавляем обработчики
        dispatcher.add_handler(conv_handler)
        dispatcher.add_handler(CommandHandler("logs", get_logs))
        dispatcher.add_handler(CommandHandler("help", help_command))

        # Запускаем бота
        updater.start_polling()
        print("Бот запущен. Для остановки нажмите Ctrl+C")

        # Держим бота работающим
        updater.idle()
    except Exception as e:
        logger.error(f"Критическая ошибка при запуске бота: {e}")
        print(f"Критическая ошибка при запуске бота: {e}")

In [ ]:
#скрипт запущен напрямую
if __name__ == "__main__":
    # Запускаем бота
    run_bot()

Архив с моделью не найден. Пожалуйста, загрузите архив qwen2.5-7b-lora-film.zip


Saving qwen2.5-7b-lora-film.zip to qwen2.5-7b-lora-film.zip
Распаковка архива с моделью...
Модель распакована успешно!
Загрузка токенизатора...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Загрузка LoRA-адаптера...
Модель успешно загружена!
Введите токен телеграм-бота: ··········
Бот запущен. Для остановки нажмите Ctrl+C
